# **1. Ollama**

**1. Installer Ollama sur Linux**

Ouvre ton terminal et tape :

```bash id="inst_ollama"
curl -fsSL https://ollama.com/install.sh | sh
```

Vérifie que ça a marché :

```bash id="verif_ollama"
ollama --version
```

---

**2. Lancer le serveur Ollama (mode API)**

Par défaut, Ollama expose son API locale sur :

```text
http://localhost:11434
```

Lance le serveur :

```bash id="serve_ollama"
ollama serve # si ça ne souvre pas ce n'est pas grave
Vérifie juste que curl http://localhost:11434 fonctionne bien
```

**Important** : laisse ce terminal ouvert.
Si tu fermes ce terminal, le serveur Ollama s’arrête.


**3. Installer un modèle gratuit**

Pour commencer avec un modèle simple mais performant :

```bash id="pull_llama3"
ollama pull llama3
```

Si tu veux **un modèle plus léger et rapide** pour CPU :

```bash id="pull_phi3"
ollama pull phi3
```

Vérifie que le modèle est bien installé :

```bash id="list_models"
ollama list
```

Exemple de sortie :

```text
NAME    ID    SIZE    MODIFIED
llama3  xxxx  4GB     2026-03-19
phi3    xxxx  2GB     2026-03-19
```


**4. Utiliser Ollama via Python (mode API)**

Crée un fichier `chat_ollama.py` avec ce code :

```python id="py_api_stream"
import requests
import json

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "llama3",  # ou "phi3" pour plus rapide
        "messages": [{"role": "user", "content": "Explique-moi la première guerre mondiale."}],
        "stream": True      # active le streaming pour voir la réponse au fur et à mesure
    },
    stream=True
)

for line in response.iter_lines():
    if line:
        chunk = json.loads(line)
        print(chunk.get("message", {}).get("content", ""), end="", flush=True)
```

Si tu veux un **JSON complet et simple** sans streaming, mets `"stream": False` et supprime `stream=True` dans `requests.post`.


**5. Comment arrêter Ollama**

Si le serveur tourne dans un terminal :

* Appuie sur `Ctrl + C` → ça arrête le serveur
* Tu peux fermer le terminal sans problème


**6. Comment relancer Ollama plus tard**

Simplement dans un nouveau terminal :

```bash id="restart_ollama"
ollama serve # Si ça ne s'ouvre ce n'est pas grave
```

* Le port reste le **11434**
* Tes modèles installés (`llama3`, `phi3`) restent **sur ton disque**, tu n’as pas besoin de les re-télécharger

---

**7. Astuces pédagogiques**

| Action                      | Commande / Astuce                                                                                             |
| --------------------------- | ------------------------------------------------------------------------------------------------------------- |
| Vérifier serveur actif      | `curl http://localhost:11434` → doit répondre “Ollama is running”                                             |
| Liste des modèles installés | `ollama list`                                                                                                 |
| Télécharger un modèle       | `ollama pull llama3` ou `ollama pull phi3`                                                                    |
| Arrêter le serveur          | `Ctrl + C`                                                                                                    |
| Relancer le serveur         | `ollama serve`                                                                                                |
| Tester un prompt rapide     | `curl -d '{"model":"llama3","messages":[{"role":"user","content":"Salut"}]}' http://localhost:11434/api/chat` |

---


Exemple des modèles gratuits populaires :

```bash id="terminal_models"
ollama pull llama3
ollama pull phi3
ollama pull mistral
ollama pull o3-mini
```



# **2. Open Router**

In [ ]:
import requests
import json

OPEN_ROUTER_TOKEN = "Bearer <your token here>"

response = requests.post(
    url="https://openrouter.ai/api/v1/chat/completions",
    headers={
        "Authorization": OPEN_ROUTER_TOKEN,
        "Content-Type": "application/json",
    },
    data=json.dumps({
        "model": "openrouter/free",
        "messages": [
            {
                "role": "system",
                "content": (
                    "Contexte : Emmanuel Macron est le président de la République française. "
                    "Il exerce actuellement son deuxième mandat présidentiel, entamé en 2022. "
                    "La durée d’un mandat présidentiel en France est de cinq ans."
                )
            },
            {
                "role": "user",
                "content": (
                    "En tenant compte uniquement du contexte fourni, "
                    "jusqu’à quelle année Emmanuel Macron exercera-t-il son mandat actuel ?"
                )
            }
        ],
    })
)

In [ ]:
response.json()

{'id': 'gen-1773920483-2rDV1n58cLQbY9Njn66Z',
 'object': 'chat.completion',
 'created': 1773920483,
 'model': 'nvidia/nemotron-3-nano-30b-a3b:free',
 'provider': 'Nvidia',
 'system_fingerprint': None,
 'choices': [{'index': 0,
   'logprobs': None,
   'finish_reason': 'stop',
   'native_finish_reason': 'stop',
   'message': {'role': 'assistant',
    'content': 'Jusqu’en\u202f2027.',
    'refusal': None,
    'reasoning': 'We need to answer based on only provided context. It says: "Emmanuel Macron est le président de la République française. Il exerce actuellement son deuxième mandat présidentiel, entamé en 2022. La durée d’un mandat présidentiel en France est de cinq ans."\n\nHe started his second term in 2022. The term length is five years. So the current term will end after five years from 2022, i.e., 2027. The question: "jusqu’à quelle année Emmanuel Macron exercera‑t‑il son mandat actuel ?" Answer: 2027 (or until end of 2027). Provide answer in French. Use only context. So answer: ju

In [ ]:


if response.status_code == 200:
    result = response.json()
    answer = result["choices"][0]["message"]["content"]
    print("Réponse du modèle :", answer)
else:
    print("Erreur :", response.status_code, response.text)



Réponse du modèle : Jusqu’en 2027.


# **3. Huggingface**

Il Faut activer d'abord le gpu

In [ ]:
# 1. Installer les dépendances (GPU obligatoire)

! pip install -U transformers accelerate bitsandbytes

In [ ]:
# 2. Importer les dépendances

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [ ]:
# 3. Définir la stratégie de quantification

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [ ]:
# 4. Charger le modèle et le tokenizer

HF_TOKEN = "" # Créer un token sur huggingface
READER_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"

model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME,
    quantization_config=bnb_config,
    token=HF_TOKEN,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(
    READER_MODEL_NAME,
    token=HF_TOKEN
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



In [ ]:
# 5. Inférence

prompt = "Explain what inference means in large language models."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

generated_text

**Exemple de sortie**

"Explain what inference means in large language models.\nInference is a process used in large language models to generate text based on a given prompt or input. It involves using the model's knowledge and understanding of language to make educated guesses about the most likely next words in a sequence, given the context and the information provided in the input. Inference is a critical component of natural language processing and is used in a wide range of applications, including language translation, speech recognition, and text generation. In large language models, inference is typically performed using a process called sampling, which involves randomly selecting a subset of the model's parameters to generate a sample of text. The quality of the generated text depends on the quality of the model's parameters and the effectiveness of the sampling process."



# **4. Gemini**

**Étapes pour récupérer ta clé**

1. Va sur :
   👉 [https://aistudio.google.com/](https://aistudio.google.com/)

2. Connecte-toi avec ton compte Google

3. Clique sur :
   **“Get API key”** (ou “Créer une clé API”)

4. Copie ta clé générée



In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key="<Your API KEY")



In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain how AI works in a few words",
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0) # Disables thinking
    ),
)
print(response.text)


AI works by recognizing patterns in data to make predictions or decisions.
